[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# El ecosistema *LangChain* v1: paquetes, modelos y salida estructurada.

**py501 · Aplicaciones Agénticas con LangChain** · Pythonista® · Instructor: José Luis Chiquete · Módulo 2: *El ecosistema LangChain v1*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se recorrerá el mapa de paquetes de *LangChain* 1.x, se inicializarán modelos de distintos proveedores con la misma interfaz, se usarán los bloques de contenido estándar y se obtendrá salida estructurada con *Pydantic*.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien quiere profundizar. Se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.
* Cada ejecución llama a *Gemini* (el modelo del taller): las respuestas varían entre ejecuciones y consumen la cuota de tu `GOOGLE_API_KEY`.

### Preparación del entorno.

* La siguiente celda localizará `data/`, cargará `.env` si existe (solo en ejecución local), definirá el modelo del taller (*Gemini*) y verificará que exista la clave `GOOGLE_API_KEY`. Si falla, sigue el *notebook* `00_claves_de_api_y_secretos.ipynb`.

In [ ]:
import os

from dotenv import load_dotenv

RAIZ = next(p for p in (os.curdir, os.pardir)
            if os.path.isdir(os.path.join(p, "data", "politicas")))
DATA = os.path.join(RAIZ, "data")
POLITICAS = os.path.join(DATA, "politicas")
load_dotenv(os.path.join(RAIZ, ".env"))   # solo tiene efecto en ejecución local; en Codespaces se usan los secretos

# Este taller usa Gemini. Cualquier proveedor funciona cambiando la cadena "proveedor:modelo" (ver notebook 03).
MODELO = os.environ.get("MODELO", "google_genai:gemini-3.8-flash")
EMBEDDINGS = os.environ.get("EMBEDDINGS", "google_genai:gemini-embedding-001")

if MODELO.startswith("google_genai") and not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Falta GOOGLE_API_KEY. Sigue el notebook 00_claves_de_api_y_secretos: guárdala como secreto de Codespaces "
                       "con acceso a este repositorio y vuelve a abrir el Codespace.")
MODELO

* La siguiente celda importará las bibliotecas de este *notebook*.

In [ ]:
from typing import Literal

from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

---
## 0. ¿Por qué *LangChain*?

Un agente se puede construir de varias formas. Todas resuelven lo mismo (llamar al modelo, ejecutar herramientas, guardar el estado y aplicar límites); cambia **cuánto de ese *harness* ya viene hecho** y con qué proveedores funciona.

| Opción | Qué es | Cuándo conviene |
|---|---|---|
| SDK del proveedor (`google-genai`, `openai`, `anthropic`) | Cliente directo de la API de un proveedor; el ciclo de herramientas lo escribes tú | Un solo proveedor y un ciclo simple |
| *OpenAI Agents SDK* | Biblioteca ligera de agentes, herramientas y traspasos entre agentes | Asistentes acotados, sobre todo con modelos de *OpenAI* |
| *Claude Agent SDK* | El *harness* de *Claude Code* como biblioteca: archivos, *shell*, permisos y subagentes | Agentes que trabajan con código o archivos, con modelos de *Anthropic* |
| *Google ADK* | Kit de Google para agentes y flujos (secuenciales, en bucle o en paralelo) | Equipos que trabajan en *Google Cloud* con *Gemini* |
| *Microsoft Agent Framework* | Framework de Microsoft para agentes y flujos de varios agentes, en *Python* y *.NET* | Equipos en el ecosistema de Microsoft |
| *LlamaIndex* | Framework centrado en datos y documentos, con flujos por eventos | Aplicaciones con mucho RAG |
| *CrewAI* | Equipos de agentes organizados por roles | Prototipos de varios agentes con roles fijos |
| *Pydantic AI* | Framework de agentes con tipos estrictos en *Python* | Equipos que priorizan tipos y pruebas |
| ***LangChain* + *LangGraph*** | `create_agent` con *middleware*, sobre el *runtime* de *LangGraph* (estado, *checkpoints* e interrupciones); independiente del proveedor; *LangSmith* para trazas y evaluación | Cambiar de proveedor sin reescribir, controlar el ciclo (aprobación humana, límites) y flujos durables |

En este taller se usa *LangChain* porque los patrones que se estudian (control del ciclo, aprobación humana, flujos durables, evaluación) vienen incluidos y no dependen del proveedor. La elección se justifica por el patrón, no por la marca: si tu caso es un ciclo simple con un solo proveedor, el SDK directo puede bastar.

* La siguiente celda hará la misma pregunta con el SDK directo de Google (`google.genai`) y con `init_chat_model`. Con el SDK, el código queda atado a un proveedor y a su formato de respuesta; con *LangChain*, el mismo código sirve para cualquier proveedor y la respuesta llega con el mismo formato (`content_blocks`, `usage_metadata`).

In [ ]:
from google import genai

PREGUNTA = "Explica en una oración qué es un agente de IA."

if MODELO.startswith("google_genai:"):
    cliente = genai.Client()   # lee GOOGLE_API_KEY del entorno
    directa = cliente.models.generate_content(model=MODELO.split(":", 1)[1], contents=PREGUNTA)
    print("SDK de Google:", directa.text)
print("LangChain:    ", init_chat_model(MODELO).invoke(PREGUNTA).text)

---
## 1. Mapa de paquetes.

| Paquete | Qué contiene | Cuándo lo importas |
|---|---|---|
| `langchain` | `create_agent`, *middleware*, `init_chat_model`, `init_embeddings`, `@tool` | Siempre: es la entrada principal |
| `langchain-core` | Interfaces base: mensajes, herramientas, *vector stores* | Para tipos y utilidades de bajo nivel |
| `langchain-<proveedor>` | Integraciones: `langchain-google-genai`, `langchain-openai`, … | Indirectamente, vía `"proveedor:modelo"` |
| `langgraph` | El *runtime*: grafos, estado, *checkpointers*, interrupciones | Cuando necesitas un flujo propio (`09`) |
| `langsmith` | Trazas, *datasets* y evaluaciones | Observabilidad y evaluación (`13`) |
| `deepagents` | Agentes con planificación, archivos y subagentes | Tareas largas (`12`) |
| `langchain-classic` | Código heredado (`AgentExecutor`, `LLMChain`) | Solo para migrar código viejo |

**La relación clave:** `create_agent` compila a un grafo de *LangGraph*, y *Deep Agents* se construye sobre `create_agent`. Siempre estás usando *LangGraph* por debajo.

* ***Middleware*:** funciones que se ejecutan antes o después de cada paso del agente para controlarlo (`08`).
* ***Vector store*:** almacén que guarda textos junto con su representación numérica y los busca por significado (`06`).

> ⚠️ **Tutoriales viejos.** Antes de la versión 1.0 se usaban `AgentExecutor`, `initialize_agent` y `langgraph.prebuilt.create_react_agent`. Todo eso es código heredado: si un tutorial lo usa, está desactualizado.

---
## 2. Modelos con una sola interfaz.

> 📌 **Modelo del taller.** En todos los *notebooks* se usa *Gemini* (`MODELO = "google_genai:gemini-3.8-flash"`). Esta sección muestra que cambiar de proveedor es cambiar esa cadena; si no tienes claves de *OpenAI* o *Anthropic*, las celdas de comparación simplemente las omiten. Para usar otro modelo en todo el taller, define la variable de entorno `MODELO` (y `EMBEDDINGS`) con el formato `"proveedor:modelo"`, por ejemplo como secreto de Codespaces.

* La siguiente celda inicializará el modelo con la cadena `"proveedor:modelo"` y le hará una pregunta.

In [ ]:
modelo = init_chat_model(MODELO)
respuesta = modelo.invoke("Explica en una oración qué es un agente de IA.")
respuesta.text

* La siguiente celda mostrará los **bloques de contenido estándar** y el uso de tokens. El formato es el mismo con cualquier proveedor.

In [ ]:
respuesta.content_blocks, respuesta.usage_metadata

**Observa:** `content_blocks` normaliza el contenido (texto, razonamiento, imágenes, *tool calls*) a un formato común. Así el código que procesa respuestas no depende del proveedor.

**Temperatura y razonamiento.**
* La **temperatura** controla cuánta variación hay al elegir cada *token*: con valores bajos, las respuestas se parecen más entre sí; con valores altos, varían más.
* En *Gemini* 3.x la temperatura **no se ajusta**: Google recomienda dejar su valor por omisión, y en `gemini-3.8-flash` el servicio la ignora.
* Lo que sí se controla es cuánto **razona** el modelo antes de responder, con `thinking_level` (`"low"`, `"medium"` o `"high"`).
* Los *tokens* de razonamiento se cobran como salida y ya están incluidos en `output_tokens`; `output_token_details` los muestra aparte.

* La siguiente celda creará el modelo con razonamiento bajo y mostrará los *tokens* de razonamiento y los de salida.

In [ ]:
razona_poco = init_chat_model(MODELO, thinking_level="low")
r_bajo = razona_poco.invoke(PREGUNTA)
{"razonamiento": r_bajo.usage_metadata.get("output_token_details", {}).get("reasoning"),
 "salida_total": r_bajo.usage_metadata["output_tokens"]}

* La siguiente celda listará los modelos para los que tienes clave configurada.

In [ ]:
import os

CANDIDATOS = {
    "google_genai:gemini-3.8-flash": "GOOGLE_API_KEY",
    "openai:gpt-5.5": "OPENAI_API_KEY",
    "anthropic:claude-sonnet-5-5": "ANTHROPIC_API_KEY",
}
disponibles = [m for m, clave in CANDIDATOS.items() if os.environ.get(clave)]
disponibles

**Modelos locales.** *LangChain* también trabaja con modelos que se ejecutan en tu propia máquina, con la misma interfaz: por ejemplo, `init_chat_model("ollama:<modelo>")`. Ejecutar un modelo local requiere un servidor como *Ollama* y el paquete de integración correspondiente (`langchain-ollama`). En este taller no se ejecuta, porque *Codespaces* no tiene GPU.

* La siguiente celda hará la misma pregunta a cada modelo disponible.

In [ ]:
for nombre in disponibles:
    r = init_chat_model(nombre).invoke("¿Cuál es el planeta más grande del sistema solar? Responde en una palabra.")
    print(f"{nombre:35} {r.text!r:20} tokens={r.usage_metadata.get('total_tokens')}")

**💡 Ideas clave de la sección 2:**
* Cambiar de proveedor es cambiar una cadena; el resto del código no cambia.
* `content_blocks` y `usage_metadata` tienen el mismo formato en todos los proveedores.

---
## 3. Salida estructurada.

Cuando el resultado lo va a consumir **código** (no una persona), se pide un objeto validado en lugar de texto libre. Con un modelo directo se usa `with_structured_output`; con un agente, el parámetro `response_format` de `create_agent` (`04`).

Hay dos estrategias: la del **proveedor** (el modelo garantiza el esquema de forma nativa) y la de **herramienta** (el esquema se presenta como una herramienta que el modelo debe llamar). *LangChain* elige automáticamente la del proveedor cuando está disponible.

* La siguiente celda definirá un esquema con *Pydantic* y clasificará una solicitud.

In [ ]:
class Clasificacion(BaseModel):
    tipo: Literal["duda_politica", "incidente", "otro"]
    prioridad: Literal["baja", "normal", "urgente"]
    resumen: str = Field(description="Resumen de la solicitud en una oración")


clasificador = init_chat_model(MODELO).with_structured_output(Clasificacion)
clasificador.invoke("Me robaron la laptop en el aeropuerto y tenía datos de clientes.")

### Ejercicio 1 · Esquema para tickets.

Define un esquema `TicketPropuesto` con los campos `titulo` (máximo 60 caracteres), `categoria` (`hardware`, `software`, `seguridad`, `rrhh` u `otro`), `prioridad` y `requiere_humano: bool`. Crea con él un modelo con salida estructurada llamado `extractor` y úsalo para procesar las tres solicitudes de la lista `SOLICITUDES`, que define la celda siguiente.

In [ ]:
SOLICITUDES = [
    "No puedo entrar a la VPN desde ayer.",
    "Quiero instalar Tableau para un análisis.",
    "Mi jefe me pidió cambiar mi horario y no estoy de acuerdo.",
]

In [ ]:
# TU CÓDIGO AQUÍ

### Ejercicio 2 · Comparar modelos.

Si tienes claves de dos proveedores, clasifica las mismas tres solicitudes con ambos. ¿Coinciden en categoría y prioridad? ¿Cuál usó menos tokens? Si solo tienes una clave, compara `gemini-3.8-flash` con otro modelo del mismo proveedor, por ejemplo `gemini-3.5-flash-lite`.

**✍️ Tu respuesta:**

_(escribe aquí)_

### Reto 1 · Procesar en lote.

Usa `extractor.batch(SOLICITUDES, config={"max_concurrency": 2})` y compara el tiempo con el procesamiento uno por uno (`%%time` o `time.perf_counter`).

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

---
## ✅ Cierre.

**Práctica:** terminar los ejercicios 1 y 2.

**Siguiente *notebook*:** `04_primer_agente_con_create_agent.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>